In [44]:
%%writefile requirements.txt
numpy==2.3.3
pandas==2.3.3
pmdarima==2.1.1
polars==1.34.0
polars-runtime-32==1.34.0
scikit-learn==1.7.2
statsmodels==0.14.5
xgboost==3.1.1
yfinance==0.2.66

Overwriting requirements.txt


In [45]:
import os

try:
    os.makedirs("src")
except FileExistsError:
    print(f"directory already exists!")

try:
    os.makedirs("workflows")
except FileExistsError:
    print(f"directory already exists!")

directory already exists!
directory already exists!


# loading the raw data from `yfinance`

this serves as the base function for loading all data

In [46]:
%%writefile src/load_data.py

"""
function written to easily load and process stock data from `yfinance`.
"""

import pandas as pd
import yfinance as yf
import polars as pl

def load_stocks(stocks: list, start: str, end: str, use_polars: bool = True):
    """load stock data from `yfinance`.

    stocks are loaded singularly, from start to end date. option to return a 
    polars dataframe or a pandas dataframe.

    Args:
        stocks (list): single-item list of stock tockers.
        start (str): first historical date.
        end (str): last historical date (up to today).
        use_polars (bool, optional): whether to return a polars dataframe or a
            pandas dataframe. defaults to true.

    Raises:
        ValueError: raised if users enter more than 1 ticker

    Returns:
        DataFrame: polars or pandas dataframe, depending on the value of
        `use_polars`.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")
    df = yf.download(stocks, start, end)
    df.index = pd.to_datetime(df.index)
    df.columns = (
        pd.MultiIndex.from_tuples(df.columns) 
        if not isinstance(df.columns, pd.MultiIndex) else df.columns
    )
    df.columns = df.columns.set_names(["Field", "Ticker"])
    df.index.name = "Date"
    df = df.apply(pd.to_numeric, errors="coerce")

    df_out = (
        df.swaplevel("Field", "Ticker", axis=1)
        .sort_index(axis=1)
        .stack("Ticker", future_stack=True)
        .reset_index()
    )

    df_out = df_out.rename(columns=str.lower)

    return pl.from_pandas(df_out) if use_polars else df_out

Overwriting src/load_data.py


# data transformations and feature engineering

In [ ]:
%%writefile src/data_etl.py

"""
set of functions to process `yfinance` data, adding lagged features and time
indicators to build the full feature space for each model.
"""

import polars as pl
from datetime import datetime, date, timedelta
import yfinance as yf
from typing import Tuple

from src.load_data import load_stocks


def prep_columns(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """prep the columns pulled from `yfinance` into a clean dataframe

    adds a 'move' column to indicate overall daily change; time-lapse columns
    lagged over 1, 7, 30 days; and rolling mean and SD values over 7 days.

    Args:
        df (pl.DataFrame): raw `yfinance` dataframe.
        col (str): which column (choose between 'close', 'open') to compute the
            lag features for.

    Returns:
        pl.DataFrame: full dataframe with lagged features of chosen column.
    """
    if col == "move":
        df = df.with_columns(
            (pl.col("close") - pl.col("open")).alias(col)
        )

    df_out =  (
        df.select(
            [
                "date",
                "ticker",
                "volume",
                col
            ]
        )
        .sort([pl.col("ticker"), pl.col("date")], descending=False)
        .with_columns(
            pl.col(col).shift(1).over("ticker").alias(f"prev1_{col}"),
            pl.col(col).shift(7).over("ticker").alias(f"prev7_{col}"),
            pl.col(col).shift(30).over("ticker").alias(f"prev30_{col}"),
        )
        .with_columns(
            pl.col(col)
            .rolling_mean(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_mean_7")
        )
        .with_columns(
            pl.col(col)
            .rolling_std(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_std_7")
        )
    )

    return df_out


def prep_data_frame(df: pl.DataFrame) -> pl.DataFrame:
    """prepare lag columns for all of 'open', 'close', and 'move'.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.

    Returns:
        pl.DataFrame: processed stock data with lag columns for all price
            indicators.
    """
    markers = ["open", "close", "move"]
    df_out = None

    for marker in markers:
        df_prep = prep_columns(df, marker)
        if df_out is None:
            df_out = df_prep
        else:
            df_out = df_prep.join(
                df_out, on=["date", "ticker", "volume"], how="inner"
            )
    
    return (
        df_out.with_columns(
            pl.col("date").dt.weekday().alias("dow")
        )
        .with_columns(
            pl.col("date").dt.month().alias("month")
        )
        .with_columns(
            pl.when(pl.col("dow").is_in([0, 4]))
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias("mon_or_fri")
        )
    )

def build_dataset(df: pl.DataFrame, label: str = "close") -> pl.DataFrame:
    """wrapper for `prep_data_frame`.

    Args:
        df (pl.DataFrame): raw `yfinance` stock dataframe.
        label (str, optional): which of 'close' or 'move' to process. Defaults
            to "close".

    Raises:
        ValueError: only accepts 'close' or 'move'.

    Returns:
        pl.DataFrame: fully processed `yfinance` data with nulls removed.
    """
    df_feat = prep_data_frame(df)

    if label == "close":
        df_feat = df_feat.with_columns(pl.col("close").shift(-1).alias("label"))
    elif label == "move":
        df_feat = df_feat.with_columns(pl.col("move").shift(-1).alias("label"))
    else:
        raise ValueError("label must be one of ['close', 'move']")
    
    return df_feat.drop_nulls()

def build_df_with_indices(
    df: pl.DataFrame, label: str, start: str, end: str
) -> pl.DataFrame:
    """process raw dataframe and add indices.

    this is originally intended to bolster the SARIMAX model by adding broad
    exogenous variables to the features space.

    Args:
        df (pl.DataFrame): raw `yfiance` stock dataframe.
        label (str): 'close' or 'move' price indicator.
        start (str): start date for index pulling from `yfinance`.
        end (str): end date for index pulling from `yfinance`.

    Returns:
        pl.DataFrame: features data with lagged columns and added index values.
    """
    indices_list = ["SPY", "QQQ", "IWM", "VXX", "UUP", "HYG", "LQD"]
    df_idx_out = None

    for idx in indices_list:
        idx_cl = idx.replace("^", "")

        df_idx = load_stocks([idx], start, end).select(
            pl.col("date"),
            pl.col("close").alias(f"{idx_cl}_close"),
            pl.col("volume").alias(f"{idx_cl}_volume")
        )

        if df_idx_out is None:
            df_idx_out = df_idx
        else:
            df_idx_out = df_idx_out.join(df_idx, on=["date"], how="inner")
    
    df_ticker = build_dataset(df, label)

    df_out = df_ticker.join(df_idx_out, on=["date"], how="inner")
    
    return df_out

class GetSectorETF:
    """simple class to infer sector ETFs"""
    def __init__(
        self,
        indexes: list,
        label: str,
        target_ticker: str
    ):
        self.SECTOR_TO_ETF = {
            "Technology": "XLK",
            "Communication Services": "XLC",
            "Financial Services": "XLF",
            "Energy": "XLE",
            "Consumer Cyclical": "XLY",
            "Consumer Defensive": "XLP",
            "Industrials": "XLI",
            "Healthcare": "XLV",
            "Real Estate": "XLRE",
            "Utilities": "XLU",
        }

        self.indexes = indexes
        self.target_ticker = target_ticker

        if label not in ["open", "close", "move"]:
            raise ValueError("label must be one of ['open', 'close', 'move]")
        else:
            self.label = label

    def extract_stock_info(self, stock_df: pl.DataFrame) -> Tuple[str, str, str]:
        """extract the ticker, earliest, and latest data from `stock_df`.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            Tuple[str, str, str]: ticker, start_date, end_date values
        """
        tickers = [stock_df.select("ticker").unique().item()]

        date_df = (
            stock_df
            .select("date")
            .unique()
            .sort(by="date", descending=True)
        )

        min_date = date_df.select("date").tail(1).item().strftime("%Y-%m-%d")
        max_date = date_df.select("date").head(1).item().strftime("%Y-%m-%d")

        return tickers, min_date, max_date

    def infer_sector_etfs(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """using tickers, extracts the close values of sector ETFs.

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: dataframe of dates, `label` values for relevant ETFs 
                for the stock's ticker, as well as `label` and values for the
                passed-in indexes.
        """
        tickers, start_date, end_date = self.extract_stock_info(stock_df)

        etfs = set()

        for t in tickers:
            info = yf.Ticker(t).info
            sector = info.get("sector")
            if sector in self.SECTOR_TO_ETF:
                etfs.add(self.SECTOR_TO_ETF[sector])
        
        ticker_list = list(etfs)
        ticker_list += self.indexes
        df_out = None

        for ticker in ticker_list:
            df_temp = load_stocks([ticker], start_date, end_date).select(
                pl.col("date"),
                pl.col(self.label).alias(f"{ticker}")
            )

            if df_out is None:
                df_out = df_temp 
            else:
                df_out = df_out.join(df_temp, on=["date"], how="inner")
        
        return df_out
    
    def build_etf_df(self, stock_df: pl.DataFrame) -> pl.DataFrame:
        """combine ETF values with stock dataframe

        Args:
            stock_df (pl.DataFrame): stock dataframe loaded from `yfinance`.

        Returns:
            pl.DataFrame: combined dataframe of `stock_df` with ETF values.
        """
        df_etf = self.infer_sector_etfs(stock_df)

        return (
            stock_df.select(
                pl.col("date"), pl.col(self.label).alias(self.target_ticker)
            )
            .join(
                df_etf, on=["date"], how="inner"
            )
        )

def compute_returns(df: pl.DataFrame) -> pl.DataFrame:
    """compute the daily return rate.

    Args:
        df (pl.DataFrame): stock dataframe loaded from `yfinance` and processed
            through `GetSectorETF`.

    Returns:
        pl.DataFrame: polars dataframe with daily returns for a target stock and
            desired indexes.
    """
    tickers = [c for c in df.columns if c != "date"]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker).pct_change().alias(f"{ticker}_return")
        )
    
    return df

def compute_return_volatility(df: pl.DataFrame, window: int) -> pd.DataFrame:
    """compute the rolling volatility (SD) of target stock and indexes.

    returns a pandas dataframe for use in prepping data for Prophet.

    Args:
        df (pl.DataFrame): polars dataframe with target stock and index prices
            and returns.
        window (int): rolling window (in days) to compute volatility.

    Returns:
        pd.DataFrame: pandas dataframe with prices, returns, and return 
            volatility.
    """
    tickers = [c for c in df.columns if c.endswith("_return")]

    for ticker in tickers:
        df = df.with_columns(
            pl.col(ticker)
            .rolling_std(window_size=window, min_samples=1)
            .alias(f"{ticker}_return_rolling_std_{window}")
        )
    
    return df.to_pandas()

def compute_rsi(df: pd.DataFrame, ticker: str, period: int) -> pd.Series:
    """add a column for RSI over a period window.

    Args:
        df (pd.DataFrame): pandas dataframe with target stock PRICES.
        ticker (str): stock to compute RSI for.
        period (int): window (days).

    Returns:
        pd.Series: pandas series to add to `df` as a column containing RSI
            values.
    """
    prices = pd.to_numeric(df[ticker], errors="coerce")
    delta = prices.diff()

    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

def compute_price_volatility(
    df: pd.DataFrame, ticker: str, period: int
) -> pd.Series:
    """add price volatility columns.

    Args:
        df (pd.DataFrame): pandas dataframe with price data.
        ticker (str): stock to compute volatility for.
        period (int): window (days).

    Returns:
        pd.Series: column indicating lagged prices.
    """
    return df[ticker].shift(period)

def build_prophet_df(
    stocks: list,
    start: str,
    end: str,
    label: str,
    indexes: list
) -> pd.DataFrame:
    """_summary_

    Args:
        stocks (list): _description_
        start (str): _description_
        end (str): _description_
        label (str): _description_
        indexes (list): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tkr = stocks[0]

    df_raw = load_stocks(stocks, start, end)
    etfs = GetSectorETF(indexes=indexes, label=label, target_ticker=tkr)
    df_etf = etfs.build_etf_df(df_raw)

    df_ret = compute_returns(df_etf)
    df_vol = compute_return_volatility(df_ret, 10)

    df_vol["rsi_7"] = compute_rsi(df_vol, tkr, 7)
    df_vol["rsi_14"] = compute_rsi(df_vol, tkr, 14)
    df_vol["rsi_21"] = compute_rsi(df_vol, tkr, 21)
    df_vol["prev7_close"] = compute_price_volatility(df_vol, tkr, 1)
    df_vol["prev14_close"] = compute_price_volatility(df_vol, tkr, 7)
    df_vol["prev30_close"] = compute_price_volatility(df_vol, tkr, 30)

    return df_vol.rename(
        columns={"date": "ds", f"{tkr}": "y"}
    ).dropna()

Overwriting src/data_etl.py


# prep the data for modeling

mostly just used for train-test splits but different models call for different  
data parsing (even if only slightly)

In [2]:
%%writefile src/model_preprocess.py

""" 
handle the creation of train-test splits for each model. not all are the same.
split for xgboost is traditional (X,y train/test tables), but for forecasting
models the split is a train/eval split without a test dataframe.

each split is done based on a cutoff datae to only allow training on past data 
and testing/eval on future data.
"""

from datetime import datetime
import polars as pl
import pandas as pd
from typing import Tuple

def train_test_split_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """do a train-test split based on a cutoff value.

    training data is all data prior to the cutoff, testing data is all data on
    or after the cutoff.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is 'y'.

    Returns:
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]: gives the
            "traditional" X_train, X_test, y_train, y_test output (akin to
            sklearn).
    """
    train = df.filter(pl.col("date") < cutoff)
    test = df.filter(pl.col("date") >= cutoff)

    X_train, X_test = (
        train.drop(label, "ticker", "date"),
        test.drop(label, "ticker", "date")
    )
    y_train, y_test = train[label], test[label]

    return X_train, X_test, y_train, y_test

def split_ar_on_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str, exog_feats: list
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """split autoregressive data on a cutoff value.

    this doesn't return unique tables for X and y and is specifically designed
    for SARIMAX. can also be used for training Prophet. a pandas dataframe is
    returned (not polars) for use in the forecasting models.

    Args:
        df (pl.DataFrame): dataframe to do the split on.
        cutoff (datetime): datetime object indicating the split date.
        label (str): label column to indicate which is the endogenous value.
        exog_feats (list): list of exogenous features for SARIMAX.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: train/eval dataframes returned as
            pandas dataframes.
    """
    train = df.filter(pl.col("date") < cutoff)
    eval = df.filter(pl.col("date") >= cutoff)

    train_pd = train.to_pandas()
    eval_pd = eval.to_pandas()

    chg_cols = [f"{label}_rolling_std_7"]

    cols_list = [label] + exog_feats + chg_cols

    train_pd = train_pd[cols_list]
    eval_pd = eval_pd[cols_list]

    return train_pd, eval_pd


Overwriting src/model_preprocess.py


leave this here for now  
trying to mess with `argparse` so it can run in the command line but i'll save  
that for last....

In [50]:
# import argparse

# ### parameters (use argparse module)

# # model hyperparameters
# DEFAULT_NUM_ESTIMATORS = 100
# DEFAULT_LEARNING_RATE = 0.01

# parser = argparse.ArgumentParser(
#     description="model hyperparameters"
# )

# parser.add_argument(
#     "-NUM_ESTIMATORS",
#     type=int,
#     default=DEFAULT_NUM_ESTIMATORS,
#     help="number of estimators"
# )
# parser.add_argument(
#     "-LEARNING_RATE",
#     type=float,
#     default=DEFAULT_LEARNING_RATE,
#     help="how fast the model learns"
# )

# # data parameters
# DEFAULT_LABEL = "move"

# parser.add_argument(
#     "-START_DATE",
#     type=str,
#     default=None,
#     help="data training start date"
# )

# parser.add_argument(
#     "-END_DATE",
#     type=str,
#     default=None,
#     help="data training end date"
# )

# parser.add_argument(
#     "-STOCKS",
#     type=list,
#     default=None,
#     help="stocks to forecast"
# )

# parser.add_argument(
#     "-LABEL",
#     type=str,
#     default=DEFAULT_LABEL,
#     help="one of 'move', 'open', 'close'; which of these values to forecast"
# )

# # cutoff value
# parser.add_argument(
#     "-CUTOFF",
#     type=datetime,
#     default=None,
#     help="cutoff value for train/test split"
# )

# # create args
# args = parser.parse_args()

# NUM_ESTIMATORS = args.NUM_ESTIMATORS
# LEARNING_RATE = args.LEARNING_RATE
# STOCKS = args.STOCKS
# START_DATE = args.START_DATE
# END_DATE = args.END_DATE
# LABEL = args.LABEL
# CUTOFF = args.CUTOFF

# NOW build the model

# training the model

currently only has `xgboost` model  
once `sarimax` is done (done researching/toying with it) i'll add to this module  
!!! `sarimax` ended up getting its own module...

In [51]:
%%writefile src/train_model.py

"""
contains a wrapper function for loading the data and training the XGBoost model.
forecasting is not done here, only model training.
"""

import polars as pl
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error
from datetime import datetime
from typing import Tuple

from src.load_data import load_stocks
from src.data_etl import *
from src.model_preprocess import train_test_split_cutoff


def train_xgb_model(
    stocks: list,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label: str,
    n_estimators: int,
    learning_rate: float,
) -> Tuple[xgb.XGBRegressor, float, pl.DataFrame, list]:
    """load raw data, preprocess, and train XGBoost model.

    Args:
        stocks (list): single-item list of stock tickers.
        start_date (str): when to start the dataframe.
        end_date (str): final date of the dataframe.
        cutoff (datetime): cutoff datetime object for train/test splits.
        label (str): label column (y).
        n_estimators (int): XGBoost `n_estimators` hyperparameter.
        learning_rate (float): XGBoost `learning_rate` hyperparameter.

    Raises:
        ValueError: cannot process more than one stock at a time.

    Returns:
        Tuple[xgb.XGBRegressor, float, pl.DataFrame, list]: XGBoost regression
            model, RMSE value, full dataframe with features, features list.
    """
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")

    df_raw = load_stocks(
        stocks=stocks,
        start=start_date,
        end=end_date,
        use_polars=True
    )

    df_feat = build_dataset(df=df_raw, label=label)

    X_train, X_test, y_train, y_test = train_test_split_cutoff(
        df=df_feat, cutoff=cutoff, label="label"
    )

    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.005,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)

    feature_cols = X_train.to_pandas().columns.tolist()

    return model, rmse, df_feat, feature_cols

Overwriting src/train_model.py


# forecasting

this predicts future unseen values (not just test data)

In [52]:
%%writefile src/forecaster.py

"""
contains a class for forecasting the future value from the trained XGBoost model.
"""

import polars as pl
import xgboost as xgb
from datetime import timedelta

from src.load_data import load_stocks
from src.data_etl import *


class XGBStockForecaster:
    """
    use the trained XGBoost model to make a forecast over a specified interval.
    """
    def __init__(
        self, model: xgb.XGBRegressor, feature_cols: list, label: str
    ):
        self.model = model
        self.feature_cols = feature_cols
        self.label = label
    
    def _predict_from_features(self, df_feat: pl.DataFrame) -> float:
        """make a prediction based on the passed in features.

        Args:
            df_feat (pl.DataFrame): features table on which the model was
                trained.

        Returns:
            float: single predicted value.
        """
        row_pd = df_feat.select(self.feature_cols).tail(1).to_pandas()
        preds = self.model.predict(row_pd)
        return float(preds[0])
    
    def forecast_horizon(self, df_raw: pl.DataFrame, days: int) -> pl.DataFrame:
        """run a forecast on the full horizon indciated by `days`.

        Args:
            df_raw (pl.DataFrame): raw `yfinance` stock dataframe.
            days (int): number of days to forecast for.

        Returns:
            pl.DataFrame: table with a date and a predicted value column.
        """
        df_current = df_raw.clone()

        forecast_dates = []
        forecast_values = []

        for _ in range(days):
            df_feat = prep_data_frame(df_current)
            pred = self._predict_from_features(df_feat)
            last_date = df_current["date"][-1]
            next_date = last_date + timedelta(days=1)

            while next_date.weekday() >= 5:
                next_date = next_date + timedelta(days=1)
            
            forecast_dates.append(next_date)
            forecast_values.append(pred)

            last_row = df_current.tail(1)

            date_dtype = df_current.schema["date"]

            new_row = last_row.with_columns(
                pl.lit(next_date).cast(date_dtype).alias("date"),
                pl.lit(pred).alias(f"{self.label}")
            )

            df_current = df_current.vstack(new_row)
        
        return pl.DataFrame(
            {
                "date": forecast_dates,
                f"pred_{self.label}": forecast_values
            }
        )

Overwriting src/forecaster.py


# full modeling pipeline

raw data -> ETL -> split -> train a model -> evaluate model training -> forecast

In [53]:
%%writefile src/pipeline.py

"""
full pipeline for loading, transforming, training, and forecasting data for the
XGBoost model.
"""

import polars as pl
from datetime import datetime
from typing import Tuple

from src.load_data import load_stocks
from src.train_model import train_xgb_model
from src.forecaster import XGBStockForecaster

def train_and_forecast_xgb(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    horizon_days: int,
    label: str = "close",
    n_estimators: int = 200,
    learning_rate: float = 0.05
) -> Tuple[pl.DataFrame, float]:
    """full pipeline for XGBoost model.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        label (str, optional): which value to predict. defaults to "close".
        n_estimators (int, optional): XGBoost `n_estimators` hyperparameter.
            defaults to 200.
        learning_rate (float, optional): XGBoost `learning_rate` hyperparameter.
            defaults to 0.05.

    Returns:
        Tuple[pl.DataFrame, float]: dataframe of predicted values per date and
            the RMSE from training.
    """
    stocks = [ticker]

    model, rmse, df_feat, feature_cols = train_xgb_model(
        stocks=stocks,
        start_date=start_date,
        end_date=end_date,
        cutoff=cutoff,
        label=label,
        n_estimators=n_estimators,
        learning_rate=learning_rate
    )

    df_raw = load_stocks(stocks, start_date, end_date)

    forecaster = XGBStockForecaster(model, feature_cols, label=label)
    forecasts_df = forecaster.forecast_horizon(df_raw, days=horizon_days)

    return forecasts_df, rmse

Overwriting src/pipeline.py


In [54]:
%%writefile src/fit_sarimax_model.py

"""
full pipeline for loading the data, training, and forecasting with SARIMAX.
"""

import polars as pl
import pandas as pd
import pmdarima as pm 
from sklearn.metrics import root_mean_squared_error
from datetime import datetime, timedelta
from typing import Tuple

from src.load_data import load_stocks
from src.data_etl import *
from src.model_preprocess import split_ar_on_cutoff
from src.utils import build_forecast_dates


def fit_sarimax(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int,
    eval_mode: bool = True
):
    """fit the actual SARIMAX model.

    has two modes: eval and forecast. eval mode uses the train-eval split to 
    gauge how accurate the forecasts are (using RMSE). forecast mode uses the
    full dataset to train and make a forecast.

    Args:
        ticker (str): stock ticker to predict.
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.
        eval_mode (bool, optional): whether to run the model as an evaluation of
            performance or as a full forecast. defaults to True (i.e., evaluate 
            the model performance).

    Returns:
        _type_: output depends on `eval_mode`. returns a float (RMSE) if 
            `eval_mode` is True, or a table (date, predicted value) if 
            `eval_mode` is False.
    """
    stock = [ticker]

    df_raw = load_stocks(stock, start_date, end_date)
    df_idx = build_df_with_indices(df_raw, label, start_date, end_date)

    feats = [
        "date",
        "dow",
        "month",
        "mon_or_fri",
        "volume",
        "SPY_close",
        "SPY_volume",
        "QQQ_close",
        "QQQ_volume",
        "IWM_close",
        "IWM_volume",
        "VXX_close",
        "VXX_volume",
        "UUP_close",
        "UUP_volume",
        "HYG_close",
        "HYG_volume",
        "LQD_close",
        "LQD_volume"
    ]

    exog_cols = [feat for feat in feats if feat != "date"]

    _sarima_hyperparams = {
        "start_p": 1,
        "start_q": 1,
        "test": "adf",
        "max_p": 3,
        "max_q": 3,
        "m": 5,
        "start_P": 0,
        "seasonal": True,
        "d": None,
        "D": None,
        "trace": False,
        "error_action": "ignore",
        "suppress_warnings": True,
        "stepwise": True
    }
    
    if eval_mode:
        df_train, df_eval = split_ar_on_cutoff(df_idx, cutoff, "close", feats)

        sarimax_model = pm.auto_arima(
            df_train[[label]],
            exogenous=df_train[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=len(df_eval),
            return_conf_int=True,
            exogenous=df_eval[exog_cols]
        )

        fitted = pd.DataFrame(fitted, columns=["pred"]).reset_index(drop=True)
        df_eval["pred"] = fitted["pred"]
        rmse = root_mean_squared_error(df_eval[["close"]], df_eval[["pred"]])
        
        return rmse
    else:
        df = df_idx.to_pandas()

        sarimax_model = pm.auto_arima(
            df[[label]],
            exogenous=df[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=horizon_days,
            return_conf_int=True,
            exogenous=df[exog_cols]
        )
        fitted = pd.DataFrame(fitted, columns=[f"pred_{label}"]).reset_index(
            drop=True
        )
        ci_series = pd.DataFrame(
            confint, columns=["lower_bound", "upper_bound"]
        )
        
        df_out = build_forecast_dates(
            end_date, horizon_days, skip_weekends=True
        )
        
        df_out[f"pred_{label}"] = fitted[f"pred_{label}"]
        df_out["lower_bound"] = ci_series["lower_bound"]
        df_out["upper_bound"] = ci_series["upper_bound"]

        return df_out.sort_values(by="date", ascending=True)

def sarimax_wrapper(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    horizon_days: int
) -> Tuple[pl.DataFrame, float]:
    """wrapper to run both versions of `fit_sarimax`.

    gets training results (`eval_mode == True`) and forecast results (`eval_mode
    == False`).

    Args:
        ticker (str): stock ticker to predict
        start_date (str): when to start the training data.
        end_date (str): last day of the training data.
        label (str): which value to predict. defaults to "close".
        cutoff (datetime): datetime object for train-test split.
        horizon_days (int): how many days in the future to forecast.

    Returns:
        Tuple[pl.DataFrame, float]: table of predictions (date, predicted value)
            and the RMSE from training.
    """
    rmse = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=True
    )

    forecasts = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        horizon_days,
        eval_mode=False
    )

    return pl.from_pandas(forecasts), rmse

Overwriting src/fit_sarimax_model.py


In [55]:
%%writefile src/utils.py

"""
utility functions
    build_forecast_dates
"""

from datetime import timedelta
import pandas as pd

def build_forecast_dates(
        last_date,
        horizon_days: int,
        skip_weekends: bool = True
) -> pd.DataFrame:
    """create a dataframe of dates based on `horizon_days`.

    Args:
        last_date (_type_): either a string date or datetime object.
        horizon_days (int): number of days (rows) to create dataframe of.
        skip_weekends (bool, optional): should weekends be skipped. defaults to
            True.

    Returns:
        pd.DataFrame: pandas dataframe of dates.
    """
    if not isinstance(last_date, (pd.Timestamp, )):
        last_date = pd.Timestamp(last_date)
    
    forecast_dates = []
    current_date = last_date 

    for _ in range(horizon_days):
        current_date = current_date + timedelta(days=1)

        if skip_weekends:
            while current_date.weekday() >= 5:
                current_date = current_date + timedelta(days=1)
        
        forecast_dates.append(current_date)
    
    return pd.DataFrame({"date": forecast_dates})

Overwriting src/utils.py


In [4]:
# %%writefile workflows/train_and_forecast_model.py

"""
script for getting all model results.
"""

from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from timeit import default_timer as timer

import warnings
warnings.filterwarnings("ignore")

from src.pipeline import train_and_forecast_xgb
from src.fit_sarimax_model import *

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - relativedelta(years=10)).strftime("%Y-%m-%d")
cutoff = (datetime.today() - relativedelta(months=2))

ticker = "MSFT"
horizon = 10

print(f"\n\nforecasting '{ticker}' prices over the next {horizon} days")
print(
    f"training models from {start_date} to {end_date}, splitting on {cutoff.strftime("%Y-%m-%d")}\n\n"
)

timer_xgb_start = timer()

xgb_forecasts, xgb_rmse = train_and_forecast_xgb(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    cutoff=cutoff,
    horizon_days=horizon,
    label="close",
    n_estimators=200,
    learning_rate=0.05
)
timer_xgb_end = timer() - timer_xgb_start

print(f"\nXGBoost test RMSE on holdout: {xgb_rmse:.4f}\n")
print(f"XGBoost run duration: {timer_xgb_end:.5f} seconds\n\n")
print(xgb_forecasts)
print("\n\n")

timer_smax_start = timer()

smax_forecasts, smax_rmse = sarimax_wrapper(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    label="close",
    cutoff=cutoff,
    horizon_days=horizon,    
)
timer_smax_end = timer() - timer_smax_start

print(f"\nSARIMAX test RMSE on holdout: {smax_rmse:.4f}\n")
print(f"SARIMAX run duration: {timer_smax_end:.5f} seconds\n\n")
print(smax_forecasts)
print("\n\n")

[*********************100%***********************]  1 of 1 completed



forecasting 'MSFT' prices over the next 10 days
training models from 2015-11-22 to 2025-11-22, splitting on 2025-09-22





[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



XGBoost test RMSE on holdout: 8.5567

XGBoost run duration: 0.54304 seconds


shape: (10, 2)
┌─────────────────────┬────────────┐
│ date                ┆ pred_close │
│ ---                 ┆ ---        │
│ datetime[μs]        ┆ f64        │
╞═════════════════════╪════════════╡
│ 2025-11-24 00:00:00 ┆ 490.076263 │
│ 2025-11-25 00:00:00 ┆ 487.280701 │
│ 2025-11-26 00:00:00 ┆ 480.947876 │
│ 2025-11-27 00:00:00 ┆ 490.374329 │
│ 2025-11-28 00:00:00 ┆ 487.396698 │
│ 2025-12-01 00:00:00 ┆ 479.815033 │
│ 2025-12-02 00:00:00 ┆ 485.415924 │
│ 2025-12-03 00:00:00 ┆ 485.633362 │
│ 2025-12-04 00:00:00 ┆ 489.828583 │
│ 2025-12-05 00:00:00 ┆ 488.974487 │
└─────────────────────┴────────────┘





[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


SARIMAX test RMSE on holdout: 14.4311

SARIMAX run duration: 5.75109 seconds


shape: (10, 4)
┌─────────────────────┬────────────┬─────────────┬─────────────┐
│ date                ┆ pred_close ┆ lower_bound ┆ upper_bound │
│ ---                 ┆ ---        ┆ ---         ┆ ---         │
│ datetime[ns]        ┆ f64        ┆ f64         ┆ f64         │
╞═════════════════════╪════════════╪═════════════╪═════════════╡
│ 2025-11-24 00:00:00 ┆ 479.239861 ┆ 470.413076  ┆ 488.066647  │
│ 2025-11-25 00:00:00 ┆ 479.440154 ┆ 467.394398  ┆ 491.48591   │
│ 2025-11-26 00:00:00 ┆ 479.640448 ┆ 465.07033   ┆ 494.210565  │
│ 2025-11-27 00:00:00 ┆ 479.840741 ┆ 463.123192  ┆ 496.558289  │
│ 2025-11-28 00:00:00 ┆ 480.041034 ┆ 461.422105  ┆ 498.659963  │
│ 2025-12-01 00:00:00 ┆ 480.241327 ┆ 459.897959  ┆ 500.584695  │
│ 2025-12-02 00:00:00 ┆ 480.441621 ┆ 458.50898   ┆ 502.374261  │
│ 2025-12-03 00:00:00 ┆ 480.641914 ┆ 457.227627  ┆ 504.0562    │
│ 2025-12-04 00:00:00 ┆ 480.842207 ┆ 456.034609  ┆ 505.64980

In [64]:
import polars as pl
import pandas as pd 
import numpy as np 
from datetime import datetime, date, timedelta 
from prophet import Prophet
from sklearn.metrics import root_mean_squared_error

import warnings
warnings.filterwarnings("ignore")

from src.load_data import load_stocks
from src.utils import build_forecast_dates
from src.data_etl import GetSectorETF, build_df_with_indices

stk = ["AAPL"]
start_date = "2015-09-01"
cutoff = date(2025, 7, 1)
end_date = "2025-09-01" 
label = "close"
idxs = ["VXX", "QQQ", "SPY", "IWM"]

df_prophet = build_prophet_df(
    stk,
    start_date,
    end_date,
    label,
    idxs
)

df_prophet.head(10)

# def build_prophet_df(
#         stocks: list,
#         start: str,
#         end: str,
#         label: str,
#         # cutoff: datetime = None,
#         # split: bool = True
# ) -> pl.DataFrame:
    
#     df = load_stocks(stocks, start, end)
#     dfi = prep_columns(df, col=label).select(
#         pl.col("date"),
#         pl.col(label),
#         pl.col(f"prev1_{label}"),
#         pl.col(f"prev7_{label}"),
#         pl.col(f"prev30_{label}")        
#     )

#     return dfi

#     if split:
#         if cutoff is None:
#             raise ValueError("provide cutoff date to perform train-eval split")
#         else:
#             df_train = dfi.filter(pl.col("ds") < cutoff)#.to_pandas()
#             df_eval = dfi.filter(pl.col("ds") >= cutoff)#.to_pandas()

#             return df_train, df_eval

#     else:
#         return dfi#.to_pandas()

# df_train, df_eval = build_prophet_df(stk, start_date, end_date, label, cutoff)
# df_full = build_prophet_df(stk, start_date, end_date, label, split=False)





# basic prophet model
# pr_mod = Prophet(interval_width=0.95)
# pr_mod.fit(df_train)

# eval_periods = len(df_eval)

# futures = pr_mod.make_future_dataframe(periods=eval_periods, freq="B")

# pr_forecast = pr_mod.predict(futures)
# pr_forecast = pr_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
# df_test = df_eval.merge(pr_forecast, on="ds", how="inner")

# rmse = root_mean_squared_error(df_test["y"], df_test["yhat"])
# print(f"\nRMSE: {rmse}\n")

# pr_mod2 = Prophet(interval_width=0.95)
# pr_mod2.fit(df_full)
# futures_final = pr_mod2.make_future_dataframe(periods=100, freq="B")
# pr_forecast_final = pr_mod2.predict(futures_final)

# import polars as pl
# import pandas as pd 
# import numpy as np 
# from datetime import datetime, date, timedelta 
# from prophet import Prophet
# from sklearn.metrics import root_mean_squared_error

# import warnings
# warnings.filterwarnings("ignore")

# from src.load_data import load_stocks
# from src.utils import build_forecast_dates
# from src.data_etl import build_df_with_indices

# stk = ["AAPL"]
# start_date = "2015-09-01"
# cutoff = date(2025, 7, 1)
# end_date = "2025-09-01" 

# def build_prophet_df(
#         stocks: list,
#         start: str,
#         end: str,
#         label: str,
#         cutoff: datetime = None,
#         split: bool = True
# ) -> pl.DataFrame:
    
#     df = load_stocks(stocks, start, end).select(
#         pl.col("date").alias("ds"),
#         pl.col(label).alias("y")
#     )

#     if split:
#         if cutoff is None:
#             raise ValueError("provide cutoff date to perform train-eval split")
#         else:
#             df_train = df.filter(pl.col("ds") < cutoff).to_pandas()
#             df_eval = df.filter(pl.col("ds") >= cutoff).to_pandas()

#             return df_train, df_eval

#     else:
#         return df.to_pandas()

# df_train, df_eval = build_prophet_df(stk, start_date, end_date, label, cutoff)
# df_full = build_prophet_df(stk, start_date, end_date, label, split=False)

# # basic prophet model
# pr_mod = Prophet(interval_width=0.95)
# pr_mod.fit(df_train)

# eval_periods = len(df_eval)

# futures = pr_mod.make_future_dataframe(periods=eval_periods + 20, freq="B")

# pr_forecast = pr_mod.predict(futures)
# pr_forecast = pr_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
# df_test = df_eval.merge(pr_forecast, on="ds", how="inner")

# rmse = root_mean_squared_error(df_test["y"], df_test["yhat"])
# print(f"\nRMSE: {rmse}\n")

# horizon = 10

# pr_mod2 = Prophet(interval_width=0.95)
# pr_mod2.fit(df_full)
# futures_final = pr_mod2.make_future_dataframe(periods=horizon, freq="B")
# pr_forecast_final = pr_mod2.predict(futures_final)

# future_forecast = (
#     pr_forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]]
#     .rename(
#         columns={
#             "ds": "date",
#             "yhat": f"pred_{label}",
#             "yhat_lower": "lower_bount",
#             "yhat_upper": "upper_bound"
#         }
#     )
#     .tail(horizon)
#     .reset_index(drop=True)
# )

# df_final = pl.from_pandas(future_forecast)
# print(df_final)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,ds,y,XLK,VXX,QQQ,SPY,IWM,AAPL_return,XLK_return,VXX_return,...,VXX_return_return_rolling_std_10,QQQ_return_return_rolling_std_10,SPY_return_return_rolling_std_10,IWM_return_return_rolling_std_10,rsi_7,rsi_14,rsi_21,prev7_close,prev14_close,prev30_close
30,2018-03-09,42.321693,65.250465,2448.000000,164.545517,246.849411,144.151505,0.017181,0.018930,-0.101902,...,0.055316,0.011479,0.010921,0.011509,70.557097,65.021454,62.506867,41.606846,41.884331,40.072559
31,2018-03-12,42.730846,65.454048,2544.639893,165.419800,246.539627,144.405487,0.009668,0.003120,0.039477,...,0.056093,0.010951,0.010348,0.011454,74.598196,67.393413,64.361290,42.321693,41.150673,40.166248
32,2018-03-13,42.319351,64.704460,2605.439941,163.167679,244.946335,143.734283,-0.009630,-0.011452,0.023893,...,0.050026,0.011139,0.009677,0.010178,64.250960,62.782294,61.166524,42.730846,41.435181,39.334869
33,2018-03-14,41.959583,64.648949,2662.399902,163.139145,243.689331,143.054001,-0.008501,-0.000858,0.021862,...,0.048473,0.010779,0.009160,0.008244,56.287330,58.982547,58.500753,42.319351,41.578648,39.103012
34,2018-03-15,42.008957,64.685974,2588.159912,162.996674,243.423782,142.337402,0.001177,0.000573,-0.027885,...,0.041241,0.008646,0.007378,0.008510,57.137927,59.346191,58.759749,41.959583,41.543381,39.210739
35,2018-03-16,41.860821,64.619133,2515.199951,162.511978,243.687790,143.135605,-0.003526,-0.001033,-0.028190,...,0.040272,0.008728,0.007341,0.007539,53.494276,57.693525,57.626745,42.008957,41.157730,39.292709
36,2018-03-19,41.221218,63.338066,2775.679932,159.044617,240.390594,141.756882,-0.015279,-0.019825,0.103562,...,0.053371,0.011100,0.008080,0.008312,40.488415,51.079406,52.994214,41.860821,41.606846,37.587791
37,2018-03-20,41.207104,63.310215,2722.560059,159.568069,240.799408,141.720581,-0.000342,-0.000440,-0.019138,...,0.053714,0.011062,0.008053,0.007677,40.236559,50.940620,52.895687,41.221218,42.321693,36.648682
38,2018-03-21,40.273571,62.929634,2623.360107,158.873276,240.337341,142.545959,-0.022655,-0.006011,-0.036436,...,0.054740,0.011088,0.008069,0.007373,27.186997,42.680578,46.846942,41.207104,42.730846,38.180309
39,2018-03-22,39.704514,61.305103,3020.800049,154.951920,234.329544,139.420258,-0.014130,-0.025815,0.151500,...,0.073279,0.013060,0.010916,0.010207,22.091622,38.574480,43.651987,40.273571,42.319351,37.362965


In [64]:
from src.load_data import load_stocks
from src.data_etl import build_df_with_indices

df = load_stocks(stk, start_date, end_date)
dfi = build_df_with_indices(df, "close", start_date, end_date)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [67]:
dfi.head(20).select(
    ["date", "ticker", "label", "volume", "prev1_close", "prev7_close", "SPY_close", "QQQ_close", "IWM_close", "VXX_close"]
)

date,ticker,label,volume,prev1_close,prev7_close,SPY_close,QQQ_close,IWM_close,VXX_close
datetime[ns],str,f64,i64,f64,f64,f64,f64,f64,f64
2018-01-25 00:00:00,"""AAPL""",40.166252,166116000,40.800907,41.262264,250.770828,159.974854,144.251266,1770.23999
2018-01-26 00:00:00,"""AAPL""",39.334869,156572000,40.072571,41.943764,253.674164,162.426483,144.768356,1770.23999
2018-01-29 00:00:00,"""AAPL""",39.103012,202561600,40.166252,41.981236,251.992233,161.637787,143.95195,1893.119995
2018-01-30 00:00:00,"""AAPL""",39.210735,184192800,39.334869,41.793884,249.407547,160.307419,142.573196,1955.199951
2018-01-31 00:00:00,"""AAPL""",39.292713,129915600,39.103012,41.451958,249.531494,160.97261,141.829391,1961.599976
…,…,…,…,…,…,…,…,…,…
2018-02-15 00:00:00,"""AAPL""",40.546341,204588800,39.356503,38.180298,241.680069,157.45668,138.60025,2656.0
2018-02-16 00:00:00,"""AAPL""",40.409962,160704400,40.678028,37.362972,241.750732,156.753525,139.090027,2695.679932
2018-02-20 00:00:00,"""AAPL""",40.226551,135722000,40.546341,36.334862,240.237228,157.067001,137.947113,2805.76001


In [2]:
from src.load_data import *
from src.data_etl import GetSectorETF
start_date = "2025-09-01"
end_date = "2025-09-10"

df_aapl = load_stocks(["AAPL"], start_date, end_date)

etfs = GetSectorETF(indexes=["VXX", "QQQ", "SPY"], label="close")

df_etf = etfs.build_etf_df(df_aapl)

/Users/ryantracy/Desktop/Python/stock_trader/stock_picker/src/load_data.py:32: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(stocks, start, end)
[*********************100%***********************]  1 of 1 completed
/Users/ryantracy/Desktop/Python/stock_trader/stock_picker/src/load_data.py:32: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(stocks, start, end)
[*********************100%***********************]  1 of 1 completed
/Users/ryantracy/Desktop/Python/stock_trader/stock_picker/src/load_data.py:32: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(stocks, start, end)
[*********************100%***********************]  1 of 1 completed
/Users/ryantracy/Desktop/Python/stock_trader/stock_picker/src/load_data.py:32: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(stocks, start, end)
[********

In [3]:
df_etf.head(20)

date,ticker,close,high,low,open,volume,XLK_volume,XLK_close,VXX_volume,VXX_close,QQQ_volume,QQQ_close,SPY_volume,SPY_close
datetime[ns],str,f64,f64,f64,f64,i64,i64,f64,i64,f64,i64,f64,i64,f64
2025-09-02 00:00:00,"""AAPL""",229.497528,230.626439,226.750191,229.027982,44075600,9113000,259.466522,10111400,37.389999,65876800,564.965027,81983500,638.499817
2025-09-03 00:00:00,"""AAPL""",238.239059,238.618696,234.133039,236.980285,66427800,7818300,261.014557,6683300,36.459999,54230200,569.409912,70820900,641.960205
2025-09-04 00:00:00,"""AAPL""",239.547775,239.667654,236.510726,238.219062,47549400,12198700,262.302917,8664300,35.380001,47526300,574.563904,65219200,647.325317
2025-09-05 00:00:00,"""AAPL""",239.45787,241.086297,238.259036,239.767568,54870400,19280900,262.522583,13420100,35.470001,68342500,575.392944,85178900,645.4505
2025-09-08 00:00:00,"""AAPL""",237.649628,239.917418,236.111111,239.068251,48999500,10278000,264.490082,4253600,34.810001,46371400,578.199707,63133100,647.036133
